In [ ]:
# Notebook setup: ensure project root is on sys.path
import os, sys
from pathlib import Path

_here = Path(".").resolve()
_root = _here.parent if _here.name == "notebooks" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("Working directory:", os.getcwd())


# 数据拆选与问题排查

这个 notebook 使用 `pandas` 对 `TV_annotation_valid_clean.csv` 做拆分、筛选和问题排查，重点检查：

- `label` 和 `boundary` 的数量是否匹配
- 是否包含 `I / C / Inter / B` 这类附加结构标签
- 是否存在重复标签或未知标签
- 哪些记录需要人工复查


In [ ]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

csv_path = Path("TV_annotation.csv")
df = pd.read_csv(csv_path)

print(f"CSV path: {csv_path.resolve()}")
print(f"shape: {df.shape}")
df.head()


CSV path: /Users/songhaifan/Documents/GitHub/music_topic/TV_annotation_valid_clean.csv
shape: (713, 15)


,folder,file_name (folderName_number),length,music_name,composer,period,boundary,label,chord,link,variation_num,instrument,applause_begin,appluse_end,note
0,Variouse_Russian,Variouse_Russian_1,12.29,Variations on a Russian theme in G major,Variouse,Romantic,0.09; 1.54; 2.47; 4.13; 5.04; 6.32; 7.25; 8.24; 10.15; 11.04; 12.20,T; V1; V2; V3; V4; V5; V6; V7; V8; V9,NaN,https://www.youtube.com/watch?v=aD7-1o2mrnk,10,violin; viola; cello,False,False,NaN
1,ARubinstein_OP88,ARubinstein_OP88_1,52.05,"Theme and Variations,Op.88",Anton Rubinstein,Romantic,0.01; 2.44; 4.26; 8.16; 11.36; 15.55; 19.31; 21.30; 25.23; 31.34; 36.36; 39.25; 41.39; 51.58,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; V11; V12,G major; G major; G major; C major; C minor; G major; G minor; E-flat major; E-flat minor; B major; B minor; D major; G major,https://www.youtube.com/watch?v=U6bPtB6Jyxk,12,piano,False,True,NaN
2,KEiges_OP36,KEiges_OP36_1,9.35,"Theme and Variations,Op.36 (Powell)",Konstantin Eiges,Modern,0.00; 1.14; 2.21; 3.27; 4.44; 5.55; 7.28; 9.31,T; V1; V2; V3; V4; V5; V6,NaN,https://www.youtube.com/watch?v=2MfGieomu-s,6,piano,False,False,NaN
3,ADvorak_OP36,ADvorak_OP36_1,13.38,"Tema con Variazioni,Op. 36",Antonín Dvořák,Romantic,0.01; 1.30; 2.28; 3.43; 6.16; 7.03; 7.49; 9.25; 10.12; 13.30,T; V1; V2; V3; V4; V5; V6; V7; V8,NaN,https://www.youtube.com/watch?v=GW9GPqas_TA,8,piano,False,False,NaN
4,ADvorak_OP36,ADvorak_OP36_2,15.30,"Tema con Variazioni,Op. 36",Antonín Dvořák,Romantic,0.04; 1.28; 2.35; 3.56; 6.44; 7.32; 8.26; 10.25; 11.26; 15.14,T; V1; V2; V3; V4; V5; V6; V7; V8,NaN,https://www.youtube.com/watch?v=a1pkCeFleF0,8,piano,False,False,NaN


In [2]:
core_pattern = re.compile(r"^(T|V[0-9]+)$")
known_extra_tags = {"I", "C", "Inter", "B"}

test_tags = ["T", "V1", "V10", "I", "C", "Inter", "B"]
[(tag, bool(core_pattern.match(tag))) for tag in test_tags]

[('T', True),
 ('V1', True),
 ('V10', True),
 ('I', False),
 ('C', False),
 ('Inter', False),
 ('B', False)]

In [3]:
def split_tokens(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(";") if item.strip()]


df_check = df.copy()
df_check["labels_list"] = df_check["label"].apply(split_tokens)
df_check["boundaries_list"] = df_check["boundary"].apply(split_tokens)

df_check["label_count"] = df_check["labels_list"].str.len()
df_check["boundary_count"] = df_check["boundaries_list"].str.len()
df_check["expected_boundary_count"] = df_check["label_count"] + 1
df_check["boundary_count_ok"] = df_check["boundary_count"] == df_check["expected_boundary_count"]

df_check["extra_tags"] = df_check["labels_list"].apply(
    lambda items: [item for item in items if not core_pattern.match(item)]
)
df_check["extra_tag_count"] = df_check["extra_tags"].str.len()
df_check["has_extra_tags"] = df_check["extra_tag_count"] > 0

df_check["known_extra_tags"] = df_check["extra_tags"].apply(
    lambda items: [item for item in items if item in known_extra_tags]
)
df_check["unknown_tags"] = df_check["extra_tags"].apply(
    lambda items: [item for item in items if item not in known_extra_tags]
)

df_check["duplicate_labels"] = df_check["labels_list"].apply(
    lambda items: len(items) != len(set(items))
)

df_check[[
    "file_name (folderName_number)",
    "label_count",
    "boundary_count",
    "expected_boundary_count",
    "boundary_count_ok",
    "extra_tags",
    "unknown_tags",
    "duplicate_labels",
]].head(10)


,file_name (folderName_number),label_count,boundary_count,expected_boundary_count,boundary_count_ok,extra_tags,unknown_tags,duplicate_labels
0,Variouse_Russian_1,10,11,11,True,[],[],False
1,ARubinstein_OP88_1,13,14,14,True,[],[],False
2,KEiges_OP36_1,7,8,8,True,[],[],False
3,ADvorak_OP36_1,9,10,10,True,[],[],False
4,ADvorak_OP36_2,9,10,10,True,[],[],False
5,ADvorak_OP36_3,9,10,10,True,[],[],False
6,ADvorak_OP36_4,9,10,10,True,[],[],False
7,SRachmaninoff_OP22_1,23,24,24,True,[],[],False
8,SRachmaninoff_OP22_2,23,24,24,True,[],[],False
9,SRachmaninoff_OP22_3,23,24,24,True,[],[],False


In [4]:
summary = pd.DataFrame(
    {
        "metric": [
            "总记录数",
            "包含 I/C/Inter/B 的记录数",
            "boundary 数量正常的记录数",
            "boundary 数量异常的记录数",
            "包含重复标签的记录数",
            "包含未知标签的记录数",
        ],
        "value": [
            len(df_check),
            int(df_check["has_extra_tags"].sum()),
            int(df_check["boundary_count_ok"].sum()),
            int((~df_check["boundary_count_ok"]).sum()),
            int(df_check["duplicate_labels"].sum()),
            int(df_check["unknown_tags"].apply(len).gt(0).sum()),
        ],
    }
)

extra_tag_counts = (
    df_check["known_extra_tags"]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("tag")
    .reset_index(name="count")
)

summary, extra_tag_counts


(                metric  value
 0                 总记录数    713
 1  包含 I/C/Inter/B 的记录数    353
 2    boundary 数量正常的记录数    692
 3    boundary 数量异常的记录数     21
 4           包含重复标签的记录数     14
 5           包含未知标签的记录数      0,
      tag  count
 0      C    311
 1      I     90
 2  Inter     10
 3      B      3)

In [5]:
extra_rows = df_check.loc[
    df_check["has_extra_tags"],
    [
        "folder",
        "file_name (folderName_number)",
        "music_name",
        "label",
        "boundary",
        "known_extra_tags",
        "unknown_tags",
        "label_count",
        "boundary_count",
        "expected_boundary_count",
        "boundary_count_ok",
    ],
].sort_values(["folder", "file_name (folderName_number)"])

extra_rows.head(30)


,folder,file_name (folderName_number),music_name,label,boundary,known_extra_tags,unknown_tags,label_count,boundary_count,expected_boundary_count,boundary_count_ok
40,AArensky_OP35a,AArensky_OP35a_1,Variations on a theme by Tchaikovsky,T; V1; V2; V3; V4; V5; V6; V7; C,0.10; 1.27; 3.07; 4.13; 6.09; 7.47; 10.49; 12.05; 13.35; 15.21,[C],[],9,10,10,True
41,AArensky_OP35a,AArensky_OP35a_2,Variations on a theme by Tchaikovsky,T; V1; V2; V3; V4; V5; V6; V7; C,0.09; 1.12; 2.31; 3.37; 4.49; 5.54; 7.34; 9.05; 10.19; 11.46,[C],[],9,10,10,True
42,AArensky_OP35a,AArensky_OP35a_3,Variations on a theme by Tchaikovsky,T; V1; V2; V3; V4; V5; V6; V7; C,0.08; 1.14; 2.39; 3.49; 5.11; 6.50; 9.16; 10.30; 11.43; 13.24,[C],[],9,10,10,True
43,AArensky_OP35a,AArensky_OP35a_4,Variations on a theme by Tchaikovsky,T; V1; V2; V3; V4; V5; V6; V7; C,0.06; 1.06; 2.18; 3.31; 4.58; 5.53; 7.10; 8.25; 9.52; 11.42,[C],[],9,10,10,True
44,AArensky_OP35a,AArensky_OP35a_5,Variations on a theme by Tchaikovsky,T; V1; V2; V3; V4; V5; V6; V7; C,0.02; 1.05; 2.31; 3.38; 5.07; 6.42; 9.01; 10.22; 11.43; 13.12,[C],[],9,10,10,True
191,ABeach_OP60,ABeach_OP60_1,"Amy Beach - Variations on Balkan Themes,Op. 60",T; V1; V2; V3; V4; V5; V6; V7; V8; C; C,0.00; 1.39; 2.51; 4.12; 5.18; 7.19; 10.33; 13.12; 15.35; 17.55; 22.58; 29.26,"[C, C]",[],11,12,12,True
192,ABeach_OP60,ABeach_OP60_3,"Amy Beach - Variations on Balkan Themes,Op. 60",T; V1; V2; V3; V4; V5; V6; V7; V8; C; C,0.01; 1.19; 2.14; 3.50; 4.32; 5.58; 9.02; 11.28; 13.04; 14.10; 18.47; 19.43,"[C, C]",[],11,12,12,True
193,ABeach_OP60,ABeach_OP60_4,"Amy Beach - Variations on Balkan Themes,Op. 60",T; V1; V2; V3; V4; V5; V6; V7; V8; V9; C; C,0.01; 1.29; 2.17; 3.15; 4.01; 5.21; 7.51; 9.59; 11.53; 13.07; 14.16; 18.26; 19.34,"[C, C]",[],12,13,13,True
194,ABeach_OP60,ABeach_OP60_5,"Amy Beach - Variations on Balkan Themes,Op. 60",T; V1; V2; V3; V4; V5; V6; V7; V8; C; C,0.00; 1.33; 2.39; 3.08; 4.13; 5.11; 6.30; 8.22; 9.18; 10.11; 13.37; 14.25,"[C, C]",[],11,12,12,True
68,ACasella_OP3,ACasella_OP3_1,"Variations sur une Chaconne,Op.3 (Gulyak)",T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.06; 0.29; 0.44; 0.58; 1.13; 1.29; 1.46; 2.15; 2.42; 3.01; 3.23; 4.17; 6.36,[C],[],12,13,13,True


In [6]:
special_problem_rows = df_check.loc[
    df_check["has_extra_tags"] & (~df_check["boundary_count_ok"]),
    [
        "folder",
        "file_name (folderName_number)",
        "music_name",
        "label",
        "boundary",
        "known_extra_tags",
        "label_count",
        "boundary_count",
        "expected_boundary_count",
    ],
].sort_values(["folder", "file_name (folderName_number)"])

print(f"包含 I/C/Inter/B 且 boundary 数量异常的记录数: {len(special_problem_rows)}")
special_problem_rows


包含 I/C/Inter/B 且 boundary 数量异常的记录数: 21


,folder,file_name (folderName_number),music_name,label,boundary,known_extra_tags,label_count,boundary_count,expected_boundary_count
690,FChopin_B12,FChopin_B12_7,"Variations sur un air national de Moore,B. 12a",I; T; V2; V3; V4; V5; C,0.01; 1.05; 2.03; 3.11; 4.09; 4.49; 5.49,"[I, C]",7,7,8
217,LBeethoven_WOO73,LBeethoven_WOO73_1,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.06; 0.43; 1.20; 1.56; 2.33; 3.10; 3.54; 4.25; 4.58; 5.49; 6.27; 9.23,[C],12,12,13
218,LBeethoven_WOO73,LBeethoven_WOO73_2,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.36; 1.13; 1.49; 2.26; 3.03; 3.47; 4.18; 4.51; 5.42; 6.20; 9.40,[C],12,12,13
219,LBeethoven_WOO73,LBeethoven_WOO73_3,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.05; 0.42; 1.15; 1.48; 2.21; 2.51; 3.30; 4.01; 4.29; 5.14; 5.58; 8.27,[C],12,12,13
220,LBeethoven_WOO73,LBeethoven_WOO73_4,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.42; 1.25; 2.09; 2.49; 3.28; 4.33; 5.07; 5.43; 6.47; 7.22; 11.03,[C],12,12,13
221,LBeethoven_WOO73,LBeethoven_WOO73_5,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.02; 0.41; 1.18; 2.04; 2.38; 3.23; 4.51; 5.26; 6.03; 7.23; 8.05; 11.11,[C],12,12,13
222,LBeethoven_WOO73,LBeethoven_WOO73_6,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.43; 1.27; 2.09; 2.47; 3.26; 4.07; 4.45; 5.24; 6.12; 6.55; 10.08,[C],12,12,13
223,LBeethoven_WOO73,LBeethoven_WOO73_7,10 Variations on La stessa la stessissima WoO 73,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 1.00; 2.18; 2.45; 3.34; 4.24; 5.46; 6.33; 7.18; 8.42; 9.25; 12.46,[C],12,12,13
413,WAMozart_K460,WAMozart_K460_7,"Mozart： 8 Variations on ＂Come un' Agnello＂,K.460",T; V1; V2; V3; V4; V5; V6; V7; V8; C,0.00; 0.46; 1.20; 2.21; 3.07; 4.04; 4.50; 5.31; 7.52; 12.32,[C],10,10,11
493,WAMozart_K573,WAMozart_K573_1,"Mozart - Variations on a Minuet by Duport,K.573",T; V1; V2; V3; V4; V5; V6; V7; V8; V9; C,0.07; 1.17; 2.22; 3.31; 4.41; 5.53; 7.17; 8.58; 10.08; 11.55; 12.42,[C],11,11,12


In [7]:
problem_rows = df_check.loc[
    ((~df_check["boundary_count_ok"])
    | (df_check["unknown_tags"].apply(len) > 0))
    & ~(df_check["known_extra_tags"].apply(lambda x: x == ["C"])),
    [
        "folder",
        "file_name (folderName_number)",
        "music_name",
        "label",
        "boundary",
        "extra_tags",
        "unknown_tags",
        "label_count",
        "boundary_count",
        "expected_boundary_count",
        "boundary_count_ok",
        "duplicate_labels",
    ],
].sort_values(
    [
        "boundary_count_ok",
        "folder",
        "file_name (folderName_number)",
    ],
    ascending=[True, True, True],
)

print(f"需要优先排查的记录数: {len(problem_rows)}")
problem_rows

需要优先排查的记录数: 1


,folder,file_name (folderName_number),music_name,label,boundary,extra_tags,unknown_tags,label_count,boundary_count,expected_boundary_count,boundary_count_ok,duplicate_labels
690,FChopin_B12,FChopin_B12_7,"Variations sur un air national de Moore,B. 12a",I; T; V2; V3; V4; V5; C,0.01; 1.05; 2.03; 3.11; 4.09; 4.49; 5.49,"[I, C]",[],7,7,8,False,False


In [8]:
# ─── Step: 删除异常记录，导出 TV_annotation_clean.csv ───
# 删除条件：boundary 数量异常 或 包含 unknown 标签

drop_mask = (
    (~df_check["boundary_count_ok"])
    | (df_check["unknown_tags"].apply(len) > 0)
)

df_clean = df[~drop_mask].copy()

out_path = Path("TV_annotation_clean.csv")
df_clean.to_csv(out_path, index=False)

print(f"原始记录: {len(df)}")
print(f"删除记录: {drop_mask.sum()}")
print(f"保留记录: {len(df_clean)}")
print(f"已保存至: {out_path.resolve()}")

# 删除的记录明细
dropped = df_check[drop_mask][[
    "file_name (folderName_number)", "label", "boundary",
    "label_count", "boundary_count", "expected_boundary_count",
    "unknown_tags"
]]
print()
print(f"删除的记录：")
dropped


原始记录: 713
删除记录: 21
保留记录: 692
已保存至: /Users/songhaifan/Documents/GitHub/music_topic/TV_annotation_clean.csv

删除的记录：


,file_name (folderName_number),label,boundary,label_count,boundary_count,expected_boundary_count,unknown_tags
217,LBeethoven_WOO73_1,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.06; 0.43; 1.20; 1.56; 2.33; 3.10; 3.54; 4.25; 4.58; 5.49; 6.27; 9.23,12,12,13,[]
218,LBeethoven_WOO73_2,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.36; 1.13; 1.49; 2.26; 3.03; 3.47; 4.18; 4.51; 5.42; 6.20; 9.40,12,12,13,[]
219,LBeethoven_WOO73_3,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.05; 0.42; 1.15; 1.48; 2.21; 2.51; 3.30; 4.01; 4.29; 5.14; 5.58; 8.27,12,12,13,[]
220,LBeethoven_WOO73_4,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.42; 1.25; 2.09; 2.49; 3.28; 4.33; 5.07; 5.43; 6.47; 7.22; 11.03,12,12,13,[]
221,LBeethoven_WOO73_5,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.02; 0.41; 1.18; 2.04; 2.38; 3.23; 4.51; 5.26; 6.03; 7.23; 8.05; 11.11,12,12,13,[]
222,LBeethoven_WOO73_6,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 0.43; 1.27; 2.09; 2.47; 3.26; 4.07; 4.45; 5.24; 6.12; 6.55; 10.08,12,12,13,[]
223,LBeethoven_WOO73_7,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; V10; C,0.00; 1.00; 2.18; 2.45; 3.34; 4.24; 5.46; 6.33; 7.18; 8.42; 9.25; 12.46,12,12,13,[]
413,WAMozart_K460_7,T; V1; V2; V3; V4; V5; V6; V7; V8; C,0.00; 0.46; 1.20; 2.21; 3.07; 4.04; 4.50; 5.31; 7.52; 12.32,10,10,11,[]
493,WAMozart_K573_1,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; C,0.07; 1.17; 2.22; 3.31; 4.41; 5.53; 7.17; 8.58; 10.08; 11.55; 12.42,11,11,12,[]
494,WAMozart_K573_2,T; V1; V2; V3; V4; V5; V6; V7; V8; V9; C,0.03; 0.38; 1.12; 1.44; 2.20; 2.59; 3.41; 4.32; 5.04; 6.33; 7.50,11,11,12,[]


In [9]:
# ─── 清洗后数据概览 ───
import pandas as pd
from pathlib import Path

df_final = pd.read_csv(Path("TV_annotation_clean.csv"))
print(f"总记录数: {len(df_final)}")
print()
print("Period 分布:")
print(df_final["period"].value_counts().to_string())
print()
print("Variation 数量分布 (variation_num):")
print(df_final["variation_num"].value_counts().sort_index().to_string())
print()
print("乐器前10:")
print(df_final["instrument"].value_counts().head(10).to_string())


总记录数: 692

Period 分布:
period
Classical    377
Romantic     279
Modern        36

Variation 数量分布 (variation_num):
variation_num
2     17
3     14
4     31
5     87
6     95
7     66
8     85
9     31
10    60
11    14
12    98
13    18
15    21
16     7
17     8
18     3
20     9
22     4
23     1
24     8
32     7
33     8

乐器前10:
instrument
piano                          461
piano-4hand                     61
Guitar                          40
cello; piano                    26
flute; piano                    22
harpsichord                     14
violin; cello; piano             9
harp                             8
flute; violin; viola; cello      7
strings                          6
